In [1]:
import pandas as pd
from pathlib import Path
import os

path = Path("data/home-credit-default-risk")
a = path.glob("*.csv",)

In [2]:
print(list(a))

[WindowsPath('data/home-credit-default-risk/application_test.csv'), WindowsPath('data/home-credit-default-risk/application_train.csv'), WindowsPath('data/home-credit-default-risk/bureau.csv'), WindowsPath('data/home-credit-default-risk/bureau_balance.csv'), WindowsPath('data/home-credit-default-risk/credit_card_balance.csv'), WindowsPath('data/home-credit-default-risk/HomeCredit_columns_description.csv'), WindowsPath('data/home-credit-default-risk/installments_payments.csv'), WindowsPath('data/home-credit-default-risk/POS_CASH_balance.csv'), WindowsPath('data/home-credit-default-risk/previous_application.csv')]


In [3]:
root = []
for i in a:
    print(i)
    root.append(i)

In [4]:
aaaa= pd.read_csv(path/"previous_application.csv")

In [5]:
aaaa.columns

Index(['SK_ID_PREV', 'SK_ID_CURR', 'NAME_CONTRACT_TYPE', 'AMT_ANNUITY',
       'AMT_APPLICATION', 'AMT_CREDIT', 'AMT_DOWN_PAYMENT', 'AMT_GOODS_PRICE',
       'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START',
       'FLAG_LAST_APPL_PER_CONTRACT', 'NFLAG_LAST_APPL_IN_DAY',
       'RATE_DOWN_PAYMENT', 'RATE_INTEREST_PRIMARY',
       'RATE_INTEREST_PRIVILEGED', 'NAME_CASH_LOAN_PURPOSE',
       'NAME_CONTRACT_STATUS', 'DAYS_DECISION', 'NAME_PAYMENT_TYPE',
       'CODE_REJECT_REASON', 'NAME_TYPE_SUITE', 'NAME_CLIENT_TYPE',
       'NAME_GOODS_CATEGORY', 'NAME_PORTFOLIO', 'NAME_PRODUCT_TYPE',
       'CHANNEL_TYPE', 'SELLERPLACE_AREA', 'NAME_SELLER_INDUSTRY',
       'CNT_PAYMENT', 'NAME_YIELD_GROUP', 'PRODUCT_COMBINATION',
       'DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE', 'DAYS_LAST_DUE_1ST_VERSION',
       'DAYS_LAST_DUE', 'DAYS_TERMINATION', 'NFLAG_INSURED_ON_APPROVAL'],
      dtype='str')

In [6]:
num = aaaa.groupby("SK_ID_CURR")[aaaa.select_dtypes("number").columns.to_list()].agg(["mean"])
# num_agg= aaaa.groupb
print(num.columns)
for c,s in num.columns:
    print(f"{c},{s}")

MultiIndex([(               'SK_ID_PREV', 'mean'),
            (               'SK_ID_CURR', 'mean'),
            (              'AMT_ANNUITY', 'mean'),
            (          'AMT_APPLICATION', 'mean'),
            (               'AMT_CREDIT', 'mean'),
            (         'AMT_DOWN_PAYMENT', 'mean'),
            (          'AMT_GOODS_PRICE', 'mean'),
            (  'HOUR_APPR_PROCESS_START', 'mean'),
            (   'NFLAG_LAST_APPL_IN_DAY', 'mean'),
            (        'RATE_DOWN_PAYMENT', 'mean'),
            (    'RATE_INTEREST_PRIMARY', 'mean'),
            ( 'RATE_INTEREST_PRIVILEGED', 'mean'),
            (            'DAYS_DECISION', 'mean'),
            (         'SELLERPLACE_AREA', 'mean'),
            (              'CNT_PAYMENT', 'mean'),
            (       'DAYS_FIRST_DRAWING', 'mean'),
            (           'DAYS_FIRST_DUE', 'mean'),
            ('DAYS_LAST_DUE_1ST_VERSION', 'mean'),
            (            'DAYS_LAST_DUE', 'mean'),
            (         'DAYS_TER

In [7]:
# print(num.columns.to_list())
# num.columns = [f"{c}_{s}".upper() for c,s in num.columns]
# print(num.reset_index(inplace=True))
# num.head()


In [8]:
from sklearn.preprocessing import OneHotEncoder

In [61]:
df = aaaa.copy()
bb = aaaa.select_dtypes("str")
ab = aaaa.select_dtypes("str").columns.to_list()
id_colmn='SK_ID_CURR'

dummies = pd.get_dummies(df[ab])
dummies[id_colmn] = df[id_colmn]
valid_dummy_cols = []
data_len = len(bb[colm].unique())
if data_len <= 2:
    matched_dummies = [c for c in dummies.columns if c.startswith(f"{colm}_")]
    valid_dummy_cols.extend(matched_dummies)

if valid_dummy_cols:
    cat_agg = dummies[[id_colmn]+valid_dummy_cols].groupby(id_colmn)[colm].agg(["mean","sum"])
    cat_agg.columns = [f"{c}_{s}".upper() for c,s in cat_agg.columns]
    cat_agg.reset_index(inplace=True)

cat_agg.columns

KeyError: 'Column not found: FLAG_LAST_APPL_PER_CONTRACT'

In [55]:
bb = aaaa.select_dtypes("str")
ab = aaaa.select_dtypes("str").columns.to_list()


a=[]

for colm in ab:
        data_len = len(bb[colm].unique())
        if data_len <= 2:
                a.append(colm)

a

['FLAG_LAST_APPL_PER_CONTRACT']

In [44]:
def aggregating_num_data(df:pd.DataFrame,
                         id_colmn : str,
                         exclude:str):
    final_df = df.copy()
    final_df.drop(exclude,axis=1,errors="ignore",inplace=True)
    num_df = final_df.select_dtypes(include="number")
    cat_df = final_df.select_dtypes(include="str")
    num_colm_df = num_df.columns.to_list()
    cat_colm_df = cat_df.columns.to_list()

    agg_data=pd.DataFrame()
    agg_data[id_colmn]=final_df[id_colmn]
    # if num_colm_df:
    #     agg_data = num_df.groupby(id_colmn).agg(["mean","sum"])
    #     agg_data.columns = [f"{c}_{s}".upper() for c,s in agg_data.columns]
    #     agg_data.reset_index(inplace=True)

    if cat_colm_df:
        for colm in cat_colm_df:
            data_len = len(cat_df[colm].unique())
            if data_len >= 4:
                dummies = pd.get_dummies(df[colm])
                dummies[id_colmn] = df[id_colmn]
                cat_agg = dummies.groupby(id_colmn).agg(["mean","sum"])
                cat_agg.columns = [f"{c}_{s}".upper() for c,s in cat_agg.columns]
                cat_agg.reset_index(inplace=True)
                agg_data = agg_data.merge(cat_agg,how="left",on=id_colmn)
            else:
                pass

            return agg_data

    return agg_data

In [45]:
aa = aggregating_num_data(aaaa,id_colmn='SK_ID_CURR',exclude='SK_ID_PREV')
aa.columns

Index(['SK_ID_CURR', 'CASH LOANS_MEAN', 'CASH LOANS_SUM',
       'CONSUMER LOANS_MEAN', 'CONSUMER LOANS_SUM', 'REVOLVING LOANS_MEAN',
       'REVOLVING LOANS_SUM', 'XNA_MEAN', 'XNA_SUM'],
      dtype='str')

In [42]:
import pandas as pd

def aggregating_num_data(df: pd.DataFrame,
                         id_colmn: str,
                         num_agg_seq,
                         colm_agg_seq,
                         exclude: str,
                         only_num=False):
    
    # 1. Copy the DataFrame safely
    final_df = df.copy()
    
    # Bug Fix 1: Add errors='ignore' so it won't crash if the exclude column isn't there
    final_df.drop(columns=[exclude], errors='ignore', inplace=True)
    
    # Bug Fix 2: Ensure id_colmn is kept in these dataframes so groupby can actually find it!
    # Also change 'str' to 'object' because Pandas stores text data as 'object' or 'category' types
    num_df = final_df.select_dtypes(include="number")
    if id_colmn not in num_df.columns and id_colmn in df.columns:
        num_df = pd.concat([df[[id_colmn]], num_df], axis=1)
        
    cat_df = final_df.select_dtypes(include=["object", "category"])
    if id_colmn not in cat_df.columns and id_colmn in df.columns:
        cat_df = pd.concat([df[[id_colmn]], cat_df], axis=1)

    # Get column lists excluding the ID column itself
    num_colm_df = [c for c in num_df.columns if c != id_colmn]
    cat_colm_df = [c for c in cat_df.columns if c != id_colmn]

    agg_data = pd.DataFrame()

    # --- Process Numerical Columns ---
    if num_colm_df:
        agg_data = num_df.groupby(id_colmn).agg(num_agg_seq)
        if not isinstance(num_agg_seq, dict):
            agg_data.columns = [f"{c}_{s}".upper() for c, s in agg_data.columns]
        agg_data.reset_index(inplace=True)

    # --- Process Categorical Columns ---
    # Bug Fix 3: checking 'len(cat_df[colm])' counts rows, not unique categories! 
    # Use '.nunique()' to find the number of unique categories instead.
    if cat_colm_df and not only_num:
        for colm in cat_colm_df:
            unique_count = cat_df[colm].nunique() 
            if unique_count <= 4:
                # Bug Fix 4: pd.get_dummies() instead of cat_df.get_dummies()
                dummies = pd.get_dummies(cat_df[colm]) 
                dummies[id_colmn] = df[id_colmn]
                
                cat_agg = dummies.groupby(id_colmn).agg(colm_agg_seq)
                cat_agg.columns = [f"{c}_{s}".upper() for c, s in cat_agg.columns]
                cat_agg.reset_index(inplace=True)
                
                if agg_data.empty:
                    agg_data = cat_agg
                else:
                    agg_data = agg_data.merge(cat_agg, how="left", on=id_colmn)
            
    # Bug Fix 5: Un-indent the return statement so it runs through ALL categories, 
    # instead of cutting off and exiting on the very first loop item.
    return agg_data


In [13]:
agg_data = aaaa.groupby('SK_ID_CURR').agg(previous_application_agg)
agg_data.columns

Index(['AMT_ANNUITY', 'AMT_APPLICATION', 'AMT_CREDIT', 'AMT_DOWN_PAYMENT',
       'AMT_GOODS_PRICE', 'NFLAG_LAST_APPL_IN_DAY', 'RATE_DOWN_PAYMENT',
       'RATE_INTEREST_PRIMARY', 'RATE_INTEREST_PRIVILEGED', 'CNT_PAYMENT',
       'DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE', 'DAYS_LAST_DUE_1ST_VERSION',
       'DAYS_LAST_DUE', 'DAYS_TERMINATION', 'NFLAG_INSURED_ON_APPROVAL'],
      dtype='str')

In [59]:
final_dfa = aaaa
final_df = final_dfa.select_dtypes("str").columns.to_list()
if final_df:
    for colm in final_df:
        a = final_dfa[colm].unique()
        if ( len(a)<= 4):
            print(colm)
        else:
            pass
# final_df.value_counts()

# if final_df:
#     needed_colm = 

NAME_CONTRACT_TYPE
FLAG_LAST_APPL_PER_CONTRACT
NAME_CONTRACT_STATUS
NAME_PAYMENT_TYPE
NAME_CLIENT_TYPE
NAME_PRODUCT_TYPE


In [ ]:
aa=(3,2)
len(aa)
if len(aa) >=3:
    print("h")
else:
    print("lesstha")

lesstha


In [88]:
isinstance(installments_payments_agg,list)

False

In [12]:
installments_payments_agg = {
    "AMT_PAYMENT" : "sum",
    "AMT_INSTALMENT" : "sum",
    "NUM_INSTALMENT_NUMBER" : "max"
}
previous_application_agg = {
    'AMT_ANNUITY' :"sum",
    'AMT_APPLICATION':"mean",
    'AMT_CREDIT':"mean",
    'AMT_DOWN_PAYMENT':"sum",
    'AMT_GOODS_PRICE':"sum",
    'NFLAG_LAST_APPL_IN_DAY':"sum",
    'RATE_DOWN_PAYMENT':"sum",
    'RATE_INTEREST_PRIMARY':"mean",
    'RATE_INTEREST_PRIVILEGED':"mean",
    'CNT_PAYMENT':"sum",
    'DAYS_FIRST_DRAWING':"mean",
    'DAYS_FIRST_DUE':"mean",
    'DAYS_LAST_DUE_1ST_VERSION':"mean",
    'DAYS_LAST_DUE':"mean",
    'DAYS_TERMINATION':"mean",
    'NFLAG_INSURED_ON_APPROVAL':"sum",    
}
credit_card_balance_agg = {
    'MONTHS_BALANCE':"mean",
    'AMT_BALANCE':"sum",
    'AMT_CREDIT_LIMIT_ACTUAL':"mean",
    'AMT_PAYMENT_CURRENT':"sum",
    'AMT_PAYMENT_TOTAL_CURRENT':"sum",
    'AMT_RECEIVABLE_PRINCIPAL':"mean"
}
bureau_agg={
    'DAYS_CREDIT':"max",
    'CREDIT_DAY_OVERDUE':"max",
    'AMT_CREDIT_MAX_OVERDUE':"max",
    'AMT_CREDIT_SUM':"mean",
    'AMT_CREDIT_SUM_DEBT':"mean",
    'AMT_CREDIT_SUM_LIMIT':"mean",
    'AMT_CREDIT_SUM_OVERDUE':"maen",
    'AMT_ANNUITY':"mean"
}
POS_CASH_balance_agg={
    'MONTHS_BALANCE':"mean",
    'CNT_INSTALMENT':"mean",
    'CNT_INSTALMENT_FUTURE':"sum"
}

In [ ]:
def aggregating_num_data(df:pd.DataFrame,
                     df_agg:dict):
    final_df = df.copy()
    final_df = final_df.select_dtypes(include="number")
    agg_data = final_df.groupby("SK_ID_CURR").agg(df_agg).reset_index()
    return agg_data
    

def merging_df(main_df:pd.DataFrame,
               agg_df):
    final_df = main_df.copy()
    if isinstance(agg_df,list):
        for i in agg_df:
            final_df = main_df.merge(i,on="SK_ID_CURR",how="left")
    else:
        final_df = main_df.merge(agg_df,on="SK_ID_CURR",how="left")

        return final_df


In [ ]:
aaa=pd.read_csv(path/"POS_CASH_balance.csv").select_dtypes("number")
aa = pd.read_csv(path/"installments_payments.csv").select_dtypes("number")
aaa=aggregating_data(aaa,POS_CASH_balance_agg)
aa=aggregating_data(aa,installments_payments_agg)
aaa.columns,aa.columns

(Index(['SK_ID_CURR', 'MONTHS_BALANCE', 'CNT_INSTALMENT',
        'CNT_INSTALMENT_FUTURE'],
       dtype='str'),
 Index(['SK_ID_CURR', 'AMT_PAYMENT', 'AMT_INSTALMENT', 'NUM_INSTALMENT_NUMBER'], dtype='str'))

In [ ]:
bb = merging_df(aaa,aa)
bb.columns

Index(['SK_ID_CURR', 'MONTHS_BALANCE', 'CNT_INSTALMENT',
       'CNT_INSTALMENT_FUTURE', 'AMT_PAYMENT', 'AMT_INSTALMENT',
       'NUM_INSTALMENT_NUMBER'],
      dtype='str')

In [ ]:
bb["SK_ID_CURR"].is_unique

True

In [ ]:
def merge():
    

In [ ]:
i =aaaa.copy()
df = i.select_dtypes("number").groupby("SK_ID_CURR").agg(["mean","sum"]).reset_index()
df.head()

SK_ID_CURR    SK_ID_PREV          AMT_ANNUITY             AMT_APPLICATION  \
                      mean      sum        mean         sum            mean   
0     100001  1.369693e+06  1369693    3951.000    3951.000        24835.50   
1     100002  1.038818e+06  1038818    9251.775    9251.775       179055.00   
2     100003  2.281150e+06  6843451   56553.990  169661.970       435436.50   
3     100004  1.564014e+06  1564014    5357.250    5357.250        24282.00   
4     100005  2.176837e+06  4353674    4813.200    4813.200        22308.75   

             AMT_CREDIT            AMT_DOWN_PAYMENT  ... DAYS_FIRST_DUE  \
         sum       mean        sum             mean  ...           mean   
0    24835.5   23787.00    23787.0           2520.0  ...   -1709.000000   
1   179055.0  179055.00   179055.0              0.0  ...    -565.000000   
2  1306309.5  484191.00  1452573.0           3442.5  ...   -1274.333333   
3    24282.0   20106.00    20106.0           4860.0  ...    -784.000000   
4    44617.5   20076.75    40153.5           4464.0  ...    -706.000000   

          DAYS_LAST_DUE_1ST_VERSION         DAYS_LAST_DUE          \
      sum                      mean     sum          mean     sum   
0 -1709.0              -1499.000000 -1499.0  -1619.000000 -1619.0   
1  -565.0                125.000000   125.0    -25.000000   -25.0   
2 -3823.0              -1004.333333 -3013.0  -1054.333333 -3163.0   
3  -784.0               -694.000000  -694.0   -724.000000  -724.0   
4  -706.0               -376.000000  -376.0   -466.000000  -466.0   

  DAYS_TERMINATION         NFLAG_INSURED_ON_APPROVAL       
              mean     sum                      mean  sum  
0     -1612.000000 -1612.0                  0.000000  0.0  
1       -17.000000   -17.0                  0.000000  0.0  
2     -1047.333333 -3142.0                  0.666667  2.0  
3      -714.000000  -714.0                  0.000000  0.0  
4      -460.000000  -460.0                  0.000000  0.0  

[5 rows x 41 columns]

In [ ]:

def ww(i):
    cat = i.select_dtypes("str").columns.to_list()
    if cat:
        a=[]
        for w in cat:
            bb = i[w].unique()
            a.append(f"{w:<30}:{len(bb):>8}")
        return(a)

ww(aaaa)

['NAME_CONTRACT_TYPE            :       4',
 'WEEKDAY_APPR_PROCESS_START    :       7',
 'FLAG_LAST_APPL_PER_CONTRACT   :       2',
 'NAME_CASH_LOAN_PURPOSE        :      25',
 'NAME_CONTRACT_STATUS          :       4',
 'NAME_PAYMENT_TYPE             :       4',
 'CODE_REJECT_REASON            :       9',
 'NAME_TYPE_SUITE               :       8',
 'NAME_CLIENT_TYPE              :       4',
 'NAME_GOODS_CATEGORY           :      28',
 'NAME_PORTFOLIO                :       5',
 'NAME_PRODUCT_TYPE             :       3',
 'CHANNEL_TYPE                  :       8',
 'NAME_SELLER_INDUSTRY          :      11',
 'NAME_YIELD_GROUP              :       5',
 'PRODUCT_COMBINATION           :      18']

In [ ]:
ww(i)

['NAME_CONTRACT_TYPE            :       4',
 'WEEKDAY_APPR_PROCESS_START    :       7',
 'FLAG_LAST_APPL_PER_CONTRACT   :       2',
 'NAME_CASH_LOAN_PURPOSE        :      25',
 'NAME_CONTRACT_STATUS          :       4',
 'NAME_PAYMENT_TYPE             :       4',
 'CODE_REJECT_REASON            :       9',
 'NAME_TYPE_SUITE               :       8',
 'NAME_CLIENT_TYPE              :       4',
 'NAME_GOODS_CATEGORY           :      28',
 'NAME_PORTFOLIO                :       5',
 'NAME_PRODUCT_TYPE             :       3',
 'CHANNEL_TYPE                  :       8',
 'NAME_SELLER_INDUSTRY          :      11',
 'NAME_YIELD_GROUP              :       5',
 'PRODUCT_COMBINATION           :      18']

In [ ]:
bb = aaaa.select_dtypes("str"),aaaa.info()

<class 'pandas.DataFrame'>
RangeIndex: 1670214 entries, 0 to 1670213
Data columns (total 37 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   SK_ID_PREV                   1670214 non-null  int64  
 1   SK_ID_CURR                   1670214 non-null  int64  
 2   NAME_CONTRACT_TYPE           1670214 non-null  str    
 3   AMT_ANNUITY                  1297979 non-null  float64
 4   AMT_APPLICATION              1670214 non-null  float64
 5   AMT_CREDIT                   1670213 non-null  float64
 6   AMT_DOWN_PAYMENT             774370 non-null   float64
 7   AMT_GOODS_PRICE              1284699 non-null  float64
 8   WEEKDAY_APPR_PROCESS_START   1670214 non-null  str    
 9   HOUR_APPR_PROCESS_START      1670214 non-null  int64  
 10  FLAG_LAST_APPL_PER_CONTRACT  1670214 non-null  str    
 11  NFLAG_LAST_APPL_IN_DAY       1670214 non-null  int64  
 12  RATE_DOWN_PAYMENT            774370 non-null   float6

In [ ]:
bb.columns

Index(['NAME_CONTRACT_TYPE', 'WEEKDAY_APPR_PROCESS_START',
       'FLAG_LAST_APPL_PER_CONTRACT', 'NAME_CASH_LOAN_PURPOSE',
       'NAME_CONTRACT_STATUS', 'NAME_PAYMENT_TYPE', 'CODE_REJECT_REASON',
       'NAME_TYPE_SUITE', 'NAME_CLIENT_TYPE', 'NAME_GOODS_CATEGORY',
       'NAME_PORTFOLIO', 'NAME_PRODUCT_TYPE', 'CHANNEL_TYPE',
       'NAME_SELLER_INDUSTRY', 'NAME_YIELD_GROUP', 'PRODUCT_COMBINATION'],
      dtype='str')

In [ ]:
a = pd.read_csv(path/"POS_CASH_balance.csv")
a.select_dtypes("number").columns.to_list()

['SK_ID_PREV',
 'SK_ID_CURR',
 'MONTHS_BALANCE',
 'CNT_INSTALMENT',
 'CNT_INSTALMENT_FUTURE',
 'SK_DPD',
 'SK_DPD_DEF']

In [ ]:
installments_payments_agg = {
    "AMT_PAYMENT" : "sum",
    "AMT_INSTALMENT" : "sum",
    "NUM_INSTALMENT_NUMBER" : "max"
}
previous_application_agg = {
    'AMT_ANNUITY' :"sum",
    'AMT_APPLICATION':"mean",
    'AMT_CREDIT':"mean",
    'AMT_DOWN_PAYMENT':"sum",
    'AMT_GOODS_PRICE':"sum",
    'NFLAG_LAST_APPL_IN_DAY':"sum",
    'NFLAG_MICRO_CASH':"mean",
    'RATE_DOWN_PAYMENT':"sum",
    'RATE_INTEREST_PRIMARY':"mean",
    'RATE_INTEREST_PRIVILEGED':"mean",
    'CNT_PAYMENT':"sum",
    'DAYS_FIRST_DRAWING':"mean",
    'DAYS_FIRST_DUE':"mean",
    'DAYS_LAST_DUE_1ST_VERSION':"mean",
    'DAYS_LAST_DUE':"mean",
    'DAYS_TERMINATION':"mean",
    'NFLAG_INSURED_ON_APPROVAL':"sum",    
}
credit_card_balance_agg = {
    'MONTHS_BALANCE':"mean",
    'AMT_BALANCE':"sum",
    'AMT_CREDIT_LIMIT_ACTUAL':"mean",
    'AMT_PAYMENT_CURRENT':"sum",
    'AMT_PAYMENT_TOTAL_CURRENT':"sum",
    'AMT_RECEIVABLE_PRINCIPAL':"mean"
}
bureau_agg={
    'DAYS_CREDIT':"max",
    'CREDIT_DAY_OVERDUE':"max",
    'AMT_CREDIT_MAX_OVERDUE':"max",
    'AMT_CREDIT_SUM':"mean",
    'AMT_CREDIT_SUM_DEBT':"mean",
    'AMT_CREDIT_SUM_LIMIT':"mean",
    'AMT_CREDIT_SUM_OVERDUE':"maen",
    'AMT_ANNUITY':"mean"
}
POS_CASH_balance_agg={
    'MONTHS_BALANCE':"mean",
    'CNT_INSTALMENT':"mean",
    'CNT_INSTALMENT_FUTURE':"sum"
}

In [ ]:
aa = pd.read_csv(path/"HomeCredit_columns_description.csv" ,encoding="latin-1")
aa["Row"].to_list()

['SK_ID_CURR',
 'TARGET',
 'NAME_CONTRACT_TYPE',
 'CODE_GENDER',
 'FLAG_OWN_CAR',
 'FLAG_OWN_REALTY',
 'CNT_CHILDREN',
 'AMT_INCOME_TOTAL',
 'AMT_CREDIT',
 'AMT_ANNUITY',
 'AMT_GOODS_PRICE',
 'NAME_TYPE_SUITE',
 'NAME_INCOME_TYPE',
 'NAME_EDUCATION_TYPE',
 'NAME_FAMILY_STATUS',
 'NAME_HOUSING_TYPE',
 'REGION_POPULATION_RELATIVE',
 'DAYS_BIRTH',
 'DAYS_EMPLOYED',
 'DAYS_REGISTRATION',
 'DAYS_ID_PUBLISH',
 'OWN_CAR_AGE',
 'FLAG_MOBIL',
 'FLAG_EMP_PHONE',
 'FLAG_WORK_PHONE',
 'FLAG_CONT_MOBILE',
 'FLAG_PHONE',
 'FLAG_EMAIL',
 'OCCUPATION_TYPE',
 'CNT_FAM_MEMBERS',
 'REGION_RATING_CLIENT',
 'REGION_RATING_CLIENT_W_CITY',
 'WEEKDAY_APPR_PROCESS_START',
 'HOUR_APPR_PROCESS_START',
 'REG_REGION_NOT_LIVE_REGION',
 'REG_REGION_NOT_WORK_REGION',
 'LIVE_REGION_NOT_WORK_REGION',
 'REG_CITY_NOT_LIVE_CITY',
 'REG_CITY_NOT_WORK_CITY',
 'LIVE_CITY_NOT_WORK_CITY',
 'ORGANIZATION_TYPE',
 'EXT_SOURCE_1',
 'EXT_SOURCE_2',
 'EXT_SOURCE_3',
 'APARTMENTS_AVG',
 'BASEMENTAREA_AVG',
 'YEARS_BEGINEXPLUATATION_A

In [ ]:
import pandas as pd

def reading_csv(file_collection):
    
    for file_path in file_collection: 

        var_name = file_path.stem
                
        globals()[var_name] = pd.read_csv(file_path, encoding="latin-1")
        
        print(f"✅ Success: Variable '{var_name}' created from {file_path.name}")


In [ ]:
def reading_csav(file):
    for i in file: 
        var_name = i.stem
        globals()[var_name] = pd.read_csv(i,encoding="latin-1")
        print(f"{var_name} store data of {i}")

In [ ]:
reading_csv(a)

KeyboardInterrupt: 

In [ ]:
reading_csav(a)

credit_card_balance store data of data\home-credit-default-risk\credit_card_balance.csv
HomeCredit_columns_description store data of data\home-credit-default-risk\HomeCredit_columns_description.csv
installments_payments store data of data\home-credit-default-risk\installments_payments.csv


KeyboardInterrupt: 

In [ ]:
# installments_payments = pd.read_csv(path/"installments_payments.csv")
installments_payments.info()

<class 'pandas.DataFrame'>
RangeIndex: 13605401 entries, 0 to 13605400
Data columns (total 8 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   SK_ID_PREV              int64  
 1   SK_ID_CURR              int64  
 2   NUM_INSTALMENT_VERSION  float64
 3   NUM_INSTALMENT_NUMBER   int64  
 4   DAYS_INSTALMENT         float64
 5   DAYS_ENTRY_PAYMENT      float64
 6   AMT_INSTALMENT          float64
 7   AMT_PAYMENT             float64
dtypes: float64(5), int64(3)
memory usage: 830.4 MB


In [ ]:
installments_payments_grouped = installments_payments.groupby(by=["SK_ID_PREV"]).agg({
    "AMT_INSTALMENT":"sum",
    "AMT_PAYMENT":"sum",
    "NUM_INSTALMENT_NUMBER":"max"
}).reset_index()

In [ ]:
installments_payments_grouped["SK_ID_PREV"].is_unique

True

In [ ]:
credit_card_balance = pd.read_csv(path/"credit_card_balance.csv")
credit_card_balance.info(),credit_card_balance["SK_ID_PREV"].is_unique

<class 'pandas.DataFrame'>
RangeIndex: 3840312 entries, 0 to 3840311
Data columns (total 23 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   SK_ID_PREV                  int64  
 1   SK_ID_CURR                  int64  
 2   MONTHS_BALANCE              int64  
 3   AMT_BALANCE                 float64
 4   AMT_CREDIT_LIMIT_ACTUAL     int64  
 5   AMT_DRAWINGS_ATM_CURRENT    float64
 6   AMT_DRAWINGS_CURRENT        float64
 7   AMT_DRAWINGS_OTHER_CURRENT  float64
 8   AMT_DRAWINGS_POS_CURRENT    float64
 9   AMT_INST_MIN_REGULARITY     float64
 10  AMT_PAYMENT_CURRENT         float64
 11  AMT_PAYMENT_TOTAL_CURRENT   float64
 12  AMT_RECEIVABLE_PRINCIPAL    float64
 13  AMT_RECIVABLE               float64
 14  AMT_TOTAL_RECEIVABLE        float64
 15  CNT_DRAWINGS_ATM_CURRENT    float64
 16  CNT_DRAWINGS_CURRENT        int64  
 17  CNT_DRAWINGS_OTHER_CURRENT  float64
 18  CNT_DRAWINGS_POS_CURRENT    float64
 19  CNT_INSTALMENT_MATURE_CUM   floa

(None, False)

In [ ]:
credit_card_balance.groupby("SK_ID_PREV")["NAME_CONTRACT_STATUS"]

In [ ]:
credit_card_balance.describe()

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,...,AMT_RECEIVABLE_PRINCIPAL,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,SK_DPD,SK_DPD_DEF
count,3.840312e+06,3.840312e+06,3.840312e+06,3.840312e+06,3.840312e+06,3.090496e+06,3.840312e+06,3.090496e+06,3.090496e+06,3.535076e+06,...,3.840312e+06,3.840312e+06,3.840312e+06,3.090496e+06,3.840312e+06,3.090496e+06,3.090496e+06,3.535076e+06,3.840312e+06,3.840312e+06
mean,1.904504e+06,2.783242e+05,-3.452192e+01,5.830016e+04,1.538080e+05,5.961325e+03,7.433388e+03,2.881696e+02,2.968805e+03,3.540204e+03,...,5.596588e+04,5.808881e+04,5.809829e+04,3.094490e-01,7.031439e-01,4.812496e-03,5.594791e-01,2.082508e+01,9.283667e+00,3.316220e-01
std,5.364695e+05,1.027045e+05,2.666775e+01,1.063070e+05,1.651457e+05,2.822569e+04,3.384608e+04,8.201989e+03,2.079689e+04,5.600154e+03,...,1.025336e+05,1.059654e+05,1.059718e+05,1.100401e+00,3.190347e+00,8.263861e-02,3.240649e+00,2.005149e+01,9.751570e+01,2.147923e+01
min,1.000018e+06,1.000060e+05,-9.600000e+01,-4.202502e+05,0.000000e+00,-6.827310e+03,-6.211620e+03,0.000000e+00,0.000000e+00,0.000000e+00,...,-4.233058e+05,-4.202502e+05,-4.202502e+05,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,1.434385e+06,1.895170e+05,-5.500000e+01,0.000000e+00,4.500000e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,4.000000e+00,0.000000e+00,0.000000e+00
50%,1.897122e+06,2.783960e+05,-2.800000e+01,0.000000e+00,1.125000e+05,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.500000e+01,0.000000e+00,0.000000e+00
75%,2.369328e+06,3.675800e+05,-1.100000e+01,8.904669e+04,1.800000e+05,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,6.633911e+03,...,8.535924e+04,8.889949e+04,8.891451e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,3.200000e+01,0.000000e+00,0.000000e+00
max,2.843496e+06,4.562500e+05,-1.000000e+00,1.505902e+06,1.350000e+06,2.115000e+06,2.287098e+06,1.529847e+06,2.239274e+06,2.028820e+05,...,1.472317e+06,1.493338e+06,1.493338e+06,5.100000e+01,1.650000e+02,1.200000e+01,1.650000e+02,1.200000e+02,3.260000e+03,3.260000e+03


In [ ]:
credit_card_balance.select_dtypes("number").groupby("SK_ID_PREV").aggregate("mean")

,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,AMT_PAYMENT_CURRENT,...,AMT_RECEIVABLE_PRINCIPAL,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,SK_DPD,SK_DPD_DEF
SK_ID_PREV,,,,,,,,,,,,,,,,,,,,,
1000018,394447.0,-4.0,74946.285000,81000.000000,5400.000000,29478.996000,0.0,24078.996000,2594.088000,5541.750000,...,72298.197000,73602.585000,73602.585000,1.200000,8.800000,0.0,7.600000,2.000000,0.000000,0.000000
1000030,361282.0,-4.5,55991.064375,81562.500000,642.857143,17257.438125,0.0,19079.929286,2078.223750,6188.631429,...,55474.453125,55935.376875,55935.376875,0.142857,5.125000,0.0,5.714286,1.875000,0.000000,0.000000
1000031,131335.0,-8.5,52394.439375,149625.000000,12115.384615,28959.615000,0.0,23527.218462,2675.300625,29543.257500,...,51402.878437,52099.970625,52099.970625,0.307692,1.312500,0.0,1.307692,3.687500,0.000000,0.000000
1000035,436351.0,-4.0,0.000000,225000.000000,NaN,0.000000,NaN,NaN,0.000000,NaN,...,0.000000,0.000000,0.000000,NaN,0.000000,NaN,NaN,0.000000,0.000000,0.000000
1000077,181153.0,-7.0,0.000000,94090.909091,NaN,0.000000,NaN,NaN,0.000000,NaN,...,0.000000,0.000000,0.000000,NaN,0.000000,NaN,NaN,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2843476,197090.0,-49.0,37937.812263,161526.315789,947.368421,947.368421,0.0,0.000000,3443.780368,3467.420053,...,35958.626053,38133.234947,38332.182316,0.021053,0.021053,0.0,0.000000,33.989474,15.031579,3.473684
2843477,168439.0,-43.0,1663.076647,15882.352941,688.235294,688.235294,0.0,0.000000,123.165000,955.265294,...,1560.223588,1644.714529,1644.714529,0.070588,0.070588,0.0,0.000000,3.928571,0.000000,0.000000
2843478,424526.0,-46.5,5111.405000,21000.000000,1000.000000,1000.000000,0.0,0.000000,455.374719,1355.795000,...,4891.073500,5097.742000,5097.742000,0.044444,0.044444,0.0,0.000000,9.494382,0.000000,0.000000


In [ ]:
installments_payments_grouped.to_csv("data_maked/installments_grouped.csv",index=False)

In [ ]:
a = installments_payments["SK_ID_PREV"].value_counts().index[0]

In [ ]:
installments_payments[installments_payments["SK_ID_PREV"] == a].sort_values("NUM_INSTALMENT_NUMBER").count()

SK_ID_PREV                293
SK_ID_CURR                293
NUM_INSTALMENT_VERSION    293
NUM_INSTALMENT_NUMBER     293
DAYS_INSTALMENT           293
DAYS_ENTRY_PAYMENT        293
AMT_INSTALMENT            293
AMT_PAYMENT               293
dtype: int64

In [ ]:
df1 = set(installments_payments["SK_ID_PREV"])
df2 = set(credit_card_balance["SK_ID_PREV"])

matching_id = len(df1.intersection(df2))
only_in_df1 = len(df1-df2)
only_in_df2 = len(df2-df1)
all_id = len(df1.union(df2))

print(len(df1))
print(len(df2))
print(f"avail id : {matching_id}")
print(f"install : {only_in_df1}")
print(f"credit : {only_in_df2}")
print(f"all id : {all_id}")

997752
104307
avail id : 72466
install : 925286
credit : 31841
all id : 1029593


In [ ]:
df.info(),dff.info(),df1.info(),df2.info(),df3.info()

NameError: name 'df' is not defined

In [ ]:
installments_payments_grouped = pd.read_csv("data_maked/installments_grouped.csv")
installments_payments_grouped.head()

,SK_ID_PREV,AMT_INSTALMENT,AMT_PAYMENT,NUM_INSTALMENT_NUMBER
0,1000001,68443.425,68443.425,2
1,1000002,37235.565,37235.565,4
2,1000003,14854.050,14854.050,3
3,1000004,33523.155,33523.155,7
4,1000005,161735.310,147021.705,10


In [ ]:
pos = pd.read_csv(path/"POS_CASH_balance.csv")
len(set(pos["S"])) , pos["S"]

936325

In [ ]:
len(set(pos["SK_ID_CURR"]))

337252